# TCR / BCR Receptor Diversity Analysis

Complete pipeline: load -> compute 6 metrics -> visualise -> compare groups -> paired chains -> HTML report.


## 0. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Ready.")

## 1. Generate or load data

Run the demo generator or point to your own CSV/TSV files.


In [ ]:
import subprocess, os
os.chdir("..")
result = subprocess.run(["python","generate_demo.py"], capture_output=True, text=True)
print(result.stdout[-1500:])

## 2. Load data

In [ ]:
from data_loader import load_multiple, validate
from pathlib import Path
paths = sorted(Path("data").glob("*.csv"))
print(f"Files: {[p.name for p in paths]}")
df = load_multiple(paths)
validate(df)
df.head(4)

## 3. Compute diversity metrics

| Metric | Formula | What it captures |
|---|---|---|
| Low-Q | Hill q=0 | Clonal richness |
| High-Q | 1/sum(pi^2) | Inverse Simpson |
| IP Slope | slope(rarefaction) | Unseen diversity |
| IPQ | AUC(rarefaction) | Sampling saturation |
| Shannon H | -sum(pi*ln pi) | Richness + evenness |
| Simpson 1-D | 1-sum(pi^2) | Clone overlap probability |

In [ ]:
from diversity_metrics import compute_all
rows = []
for s, grp in df.groupby("sample_id"):
    rows.append(compute_all(grp["frequency"].values, sample_id=s, steps=50))
summary = pd.DataFrame(rows)
cols = ["sample","n_clones","n_reads","low_q","high_q","ip_slope","ipq","shannon_H","simpson_1minusD"]
summary[cols].round(3)

## 4. Rarefaction curves and Hill diversity profile

In [ ]:
from plots import plot_rarefaction, plot_hill_profile
fig, _ = plot_rarefaction(df, steps=40)
plt.tight_layout(); plt.show()

In [ ]:
fig, _ = plot_hill_profile(df)
plt.tight_layout(); plt.show()

## 5. V-gene usage and CDR3 lengths

In [ ]:
from plots import plot_vgene_usage, plot_cdr3_length
fig, _ = plot_vgene_usage(df, top_n=12)
if fig: plt.tight_layout(); plt.show()
fig, _ = plot_cdr3_length(df)
if fig: plt.tight_layout(); plt.show()

## 6. Statistical group comparison

- Permutation test (non-parametric)
- Mann-Whitney U + BH FDR correction
- Effect size: rank-biserial r
- Stars: * p<0.05  ** p<0.01  *** p<0.001

In [ ]:
from stats_compare import GroupComparison
groups = {
    "Healthy": [s for s in summary["sample"] if "Healthy" in s],
    "Patient": [s for s in summary["sample"] if "Patient" in s],
}
cmp = GroupComparison(summary, groups, n_perm=999)
results = cmp.run_all()
results[["group1","group2","metric","mean1","mean2","p_perm_adj","sig","effect_r"]].sort_values("p_perm_adj").round(4)

In [ ]:
path = cmp.plot_comparison(output_dir="output/")
from IPython.display import Image
Image(str(path))

In [ ]:
print("Mean +/- std per group:")
display(cmp.summary_table())

In [ ]:
kw = GroupComparison(summary,{
    "Healthy":[s for s in summary["sample"] if "Healthy" in s],
    "Patient":[s for s in summary["sample"] if "Patient" in s],
    "Recovery":[s for s in summary["sample"] if "Recovery" in s],
}, n_perm=499).kruskal_wallis()
kw

## 7. Paired alpha/beta chain analysis

For paired (single-cell or barcode-resolved bulk) TCR/BCR data.

In [ ]:
rng = np.random.default_rng(42)
vg_a = [f"TRAV{i}" for i in range(1,22)]
vg_b = [f"TRBV{i}" for i in range(1,16)]
aa = list("ACDEFGHIKLMNPQRSTVWY")
rows2 = []
for sample in ["Healthy_T","Patient_T"]:
    for ci in range(300):
        cid = f"{sample}_c{ci:04d}"
        if sample=="Patient_T" and ci<80:
            va,vb="TRAV14","TRBV12"
        else:
            va,vb=rng.choice(vg_a),rng.choice(vg_b)
        for chain,vg in [("TRA",va),("TRB",vb)]:
            rows2.append({"cell_id":cid,"chain":chain,"v_gene":vg,
                          "frequency":int(rng.integers(1,60)),
                          "cdr3_aa":"".join(rng.choice(aa,size=int(rng.integers(10,18)))),
                          "sample_id":sample})
paired_raw = pd.DataFrame(rows2)
print(paired_raw.groupby(["sample_id","chain"]).size().to_string())

In [ ]:
from paired_chains import pair_chains, per_chain_diversity, plot_vgene_pairing_heatmap, plot_cdr3_length_correlation, plot_per_chain_diversity, find_public_clones
paired = pair_chains(paired_raw, "TRA", "TRB", cell_col="cell_id")
print(f"Paired cells: {len(paired):,}")
paired.head(4)

In [ ]:
from IPython.display import Image, display
for s in paired["sample_id"].unique():
    p = plot_vgene_pairing_heatmap(paired, top_n=10, output_dir="output/", sample_id=s)
    if p: display(Image(str(p)))

In [ ]:
p = plot_cdr3_length_correlation(paired, output_dir="output/")
if p:
    from IPython.display import Image; display(Image(str(p)))

In [ ]:
chain_div = per_chain_diversity(paired_raw)
display(chain_div[["sample_id","chain","n_clones","low_q","high_q","shannon_H"]].round(3))
p = plot_per_chain_diversity(chain_div, output_dir="output/")
from IPython.display import Image; Image(str(p))

In [ ]:
public = find_public_clones(paired_raw, sequence_col="cdr3_aa", min_samples=2)
print(f"Shared clones: {len(public)}")
public.head(8)

## 8. Export HTML report

In [ ]:
from analyse import run_analysis
summary_full = run_analysis(df, output_dir="output/", verbose=True)
print("Report: output/diversity_report.html")

In [ ]:
from IPython.display import IFrame
IFrame("output/diversity_report.html", width="100%", height=700)

## 9. Using your own data

**Minimal CSV:**


**Full CSV:**


**CLI:**


**MiXCR output:**  column names  are all auto-resolved.

**VDJtools output:**  column names  are auto-resolved.